# Descrição geral
Os dados foram sintetisados usando um script simples, aqui o foco é gerar um conjunto mais simples, mais rapido e mais facil de usar no treinamento

In [ ]:
import pandas as pd
from seeds.seed_perfil_usuario_kmeans import gerar_tabela
df_perfil_usuario = gerar_tabela(numero_linhas_iguais=6000, numero_linhas_abaixo=2000, numero_linhas_acima=2000)

In [ ]:
print('Trechos')
print(df_perfil_usuario.head())

print('Shape')
print(df_perfil_usuario.shape)

print('Potenciais falhas')
print(df_perfil_usuario.isnull().sum())

# Extrair valores importante

In [ ]:
colunas_poupanca = ['POUPANCA']
colunas_gastos = [c for c in df_perfil_usuario.columns if c not in colunas_poupanca]

df_gasto_resumo = pd.DataFrame({
    'TOTAL_GASTOS': df_perfil_usuario[colunas_gastos].sum(axis=1),
    'POUPANCA': df_perfil_usuario['POUPANCA'],
})
df_gasto_resumo['TOTAL'] = df_gasto_resumo['TOTAL_GASTOS'] + df_gasto_resumo['POUPANCA']


# Normalizando valores

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

scaler = StandardScaler()

# fit + transform: aprende a média/desvio e aplica
X_normalizado = scaler.fit_transform(df_gasto_resumo)

# X_normalizado é um numpy array, média 0 e desvio 1 em cada coluna
print("Média após normalização:", X_normalizado.mean(axis=0).round(2))
print("Desvio após normalização:", X_normalizado.std(axis=0).round(2))

df_normalizado = pd.DataFrame(X_normalizado, columns=df_gasto_resumo.columns.to_list())
print(df_normalizado.head())

# Escolher quantidade de grupos

A quantidade de grupos já foi escolhida no primeiro treino, a celula vai continuar no codigo para mostrar que o valor escolhido não foi aleatorio

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inercias = []
grupos_teste = range(2, 21)

for k in grupos_teste:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_normalizado)
    inercias.append(kmeans.inertia_)
    print(f"Grupos: {k} | Inércia: {kmeans.inertia_:.2f}")

plt.figure(figsize=(8, 5))
plt.plot(grupos_teste, inercias, marker='o', linestyle='-', color='steelblue')
plt.title('Método do Cotovelo')
plt.xlabel('Número de grupos (k)')
plt.ylabel('Inércia (WCSS)')
plt.xticks(grupos_teste)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Treino do modelo
aqui é onde o filho chora e a mãe não vê

In [ ]:
from sklearn.cluster import KMeans
import pandas as pd


kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
kmeans.fit(X_normalizado)

print(f"Modelo treinado com {kmeans.n_clusters} grupos")
print(f"Inércia final: {kmeans.inertia_:.2f}")

# Desnormalizar os centros para ler em percentuais
centros = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=df_normalizado.columns.to_list()
)

print("\n" + "="*70)
print("CENTROS DOS PERFIS (valores em %)")
print("="*70)
print(centros.round(2))


In [ ]:
rotulos = kmeans.labels_
contagem = pd.Series(rotulos).value_counts().sort_index()
print("\n" + "="*70)
print("USUÁRIOS POR PERFIL")
print("="*70)
print(contagem)


In [ ]:
def prever_perfil(total_gastos: float, poupanca: float) -> dict:
    PERFIS = {
        0: "Poupador moderado",
        1: "Endividado/Consumista",
        2: "Moderado com sobra",
        3: "Superpoupador",
        4: "Gastador",
        5: "Baixa renda/Parcial"
    }
    entrada = pd.DataFrame([[total_gastos, poupanca]], columns=["TOTAL_GASTOS", "POUPANCA"])
    entrada["TOTAL"] = entrada["TOTAL_GASTOS"] + entrada["POUPANCA"]
    
    # Normaliza com o scaler treinado
    entrada_norm = scaler.transform(entrada)
    
    # Prediz o cluster
    cluster = int(kmeans.predict(entrada_norm)[0])
    
    return {
        "cluster": cluster,
        "perfil": PERFIS.get(cluster, "Desconhecido"),
    }

print(
    prever_perfil(100.0, 20)
)

# Exportar modelo


In [ ]:
import joblib

joblib.dump(kmeans, "modelos/kmeans_perfil.pkl")
joblib.dump(scaler, "modelos/scaler_perfil.pkl")